# Census data cleaner

This notebook filters the Census 2021 data, to only keep data for Brussels and groups them as needed.

The output is saved in the form of CSV files that are then used by 1_2_population_generation.ipynb.

In [ ]:
# For all files, get only data for Brussels (filter on province code) and aggregate to get at the same levels
# Source: https://statbel.fgov.be/fr/open-data/consultez-tous-les-open-data-du-census-2021
# H, M, L corresponds to the level of detail high, medium, low

# Statistical sector level
# S01: sex, age (M), sector
# S11: sex, education, sector
# S12: sex, employment (H), sector --> get employment on L level

# Active population
# S13: sex, professional status, sector
# S14: sex, work sector (H), sector --> get work sector to L level

# Province level
# HC04_3: sex, age (H), education --> get age on M level
# HC04_1: sex, age (H), employment (H) --> get age on M level + get employment on L level
# HC23_2: sex, age (M), education, employment (L), position in family --> sum over position in family

# Active population
# HC05_6: sex, age (M), professional status, work sector (L)
# HC05_7: sex, age (L), education, work sector (L)

In [ ]:
import pandas as pd

# S01: sex, age (M), sector

In [ ]:
# S01
# Filter to get only data for Brussels

# Load S01 data
s01_file = "input_data/TF_CENSUS_2021_S01.xlsx"
df_s01 = pd.read_excel(s01_file)

# Display columns to verify
print("Available columns in S01:")
print(df_s01.columns.tolist())
print(f"\nTotal records: {len(df_s01)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_s01 = df_s01[df_s01["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_s01)}")

# Keep only sector, age, sex fields
# Mapping: CD_SECTOR for sector, CD_AGE for age, CD_SEX for sex
df_cleaned = df_brussels_s01[["CD_SECTOR", "CD_AGE", "CD_SEX", "MS_POP"]].copy()

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_S01_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# S11: sex, education, sector

In [ ]:
# S11
# Filter to get only data for Brussels

# Load S11 data
s11_file = "input_data/TF_CENSUS_2021_S11.xlsx"
df_s11 = pd.read_excel(s11_file)

# Display columns to verify
print("Available columns in S11:")
print(df_s11.columns.tolist())
print(f"\nTotal records: {len(df_s11)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_s11 = df_s11[df_s11["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_s11)}")

# Keep only sector, education, sex fields
# Mapping: CD_SECTOR for sector, CD_EDU for education, CD_SEX for sex
df_cleaned = df_brussels_s11[["CD_SECTOR", "CD_EDU", "CD_SEX", "MS_POP"]].copy()

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_S11_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# S12: sex, employment (H), sector

In [ ]:
# S12
# Filter to get only data for Brussels + group employment on L level

# Load S12 data
s12_file = "input_data/TF_CENSUS_2021_S12.xlsx"
df_s12 = pd.read_excel(s12_file)

# Display columns to verify
print("Available columns in S12:")
print(df_s12.columns.tolist())
print(f"\nTotal records: {len(df_s12)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_s12 = df_s12[df_s12["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_s12)}")

# Create aggregated employment field
# If ACT: keep EMP or UNE from CD_CAS_LVL_2
# If INAC: group all as INAC
df_brussels_s12["CD_EMPLOYMENT"] = df_brussels_s12.apply(
    lambda row: row["CD_CAS_LVL_2"] if row["CD_CAS_LVL_1"] == "ACT" else "INAC",
    axis=1
)

print("\nUnique employment values after aggregation:")
print(df_brussels_s12["CD_EMPLOYMENT"].unique())

# Group by sector, sex, and employment, summing population
df_cleaned = df_brussels_s12.groupby(
    ["CD_SECTOR", "CD_SEX", "CD_EMPLOYMENT"], as_index=False
)["MS_POP"].sum()

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_S12_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# S13: sex, professional status, sector

In [ ]:
# S13
# Filter to get only data for Brussels

# Load S13 data
s13_file = "input_data/TF_CENSUS_2021_S13.xlsx"
df_s13 = pd.read_excel(s13_file)

# Display columns to verify
print("Available columns in S13:")
print(df_s13.columns.tolist())
print(f"\nTotal records: {len(df_s13)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_s13 = df_s13[df_s13["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_s13)}")

# Keep only sector, age, sex fields
# Mapping: CD_SECTOR for sector, CD_SIE for professional status, CD_SEX for sex
df_cleaned = df_brussels_s13[["CD_SECTOR", "CD_SIE", "CD_SEX", "MS_POP"]].copy()

# Rename professional status column to CD_PROF_STATUS for consistency
df_cleaned.rename(columns={"CD_SIE": "CD_PROF_STATUS"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_S13_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# S14: sex, work sector (H), sector

In [ ]:
# S14
# Filter to get only data for Brussels + group work sector on L level

# Load S14 data
s14_file = "input_data/TF_CENSUS_2021_S14.xlsx"
df_s14 = pd.read_excel(s14_file)

# Display columns to verify
print("Available columns in S14:")
print(df_s14.columns.tolist())
print(f"\nTotal records: {len(df_s14)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_s14 = df_s14[df_s14["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_s14)}")

# Group by sector, industry, and sex, summing population
df_cleaned = df_brussels_s14.groupby(
    ["CD_SECTOR", "CD_IND_LVL_1", "CD_SEX"], as_index=False
)["MS_POP"].sum()

# Rename industry column to CD_INDUSTRY for consistency
df_cleaned.rename(columns={"CD_IND_LVL_1": "CD_INDUSTRY"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_S14_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# HC04_3: sex, age (H), education

In [ ]:
# HC04_3
# Filter to get only data for Brussels + group age on M level (5-year periods)

# Load HC04_3 data
hc04_3_file = "input_data/TF_CENSUS_2021_HC04_3.xlsx"
df_hc04_3 = pd.read_excel(hc04_3_file)

# Display columns to verify
print("Available columns in HC04_3:")
print(df_hc04_3.columns.tolist())
print(f"\nTotal records: {len(df_hc04_3)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_hc04_3 = df_hc04_3[df_hc04_3["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_hc04_3)}")

print("\nUnique age values before aggregation:")
print(sorted(df_brussels_hc04_3["CD_AGE"].unique()))

# Create 5-year age groups
def age_to_5year_group(age):
    if age == "100+":
        return "100+"
    age_int = int(age)
    lower = (age_int // 5) * 5
    upper = lower + 4
    if lower >= 100:
        return "100+"
    return f"{lower}-{upper}"

df_brussels_hc04_3["CD_AGE_GROUP"] = df_brussels_hc04_3["CD_AGE"].apply(age_to_5year_group)

print("\nUnique age groups after aggregation:")
print(sorted(df_brussels_hc04_3["CD_AGE_GROUP"].unique()))

# Group by sex, education, age group, summing population
df_cleaned = df_brussels_hc04_3.groupby(
    ["CD_SEX", "CD_EDU", "CD_AGE_GROUP"], as_index=False
)["MS_POP"].sum()

# Rename age group column to CD_AGE for consistency
df_cleaned.rename(columns={"CD_AGE_GROUP": "CD_AGE"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_HC04_3_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# HC04_1: sex, age (H), employment (H)

In [ ]:
# HC04_1
# Filter to get only data for Brussels + group age on M level (5-year periods) + get employment on L level

# Load HC04_1 data
hc04_1_file = "input_data/TF_CENSUS_2021_HC04_1.xlsx"
df_hc04_1 = pd.read_excel(hc04_1_file)

# Display columns to verify
print("Available columns in HC04_1:")
print(df_hc04_1.columns.tolist())
print(f"\nTotal records: {len(df_hc04_1)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_hc04_1 = df_hc04_1[df_hc04_1["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_hc04_1)}")

print("\nUnique age values before aggregation:")
print(sorted(df_brussels_hc04_1["CD_AGE"].unique()))

# Create 5-year age groups
def age_to_5year_group(age):
    if age == "100+":
        return "100+"
    age_int = int(age)
    lower = (age_int // 5) * 5
    upper = lower + 4
    if lower >= 100:
        return "100+"
    return f"{lower}-{upper}"

df_brussels_hc04_1["CD_AGE_GROUP"] = df_brussels_hc04_1["CD_AGE"].apply(age_to_5year_group)

print("\nUnique age groups after aggregation:")
print(sorted(df_brussels_hc04_1["CD_AGE_GROUP"].unique()))

# Create aggregated employment field
# If ACT: keep EMP or UNE from CD_CAS_LVL_2
# If INAC: group all as INAC
df_brussels_hc04_1["CD_EMPLOYMENT"] = df_brussels_hc04_1.apply(
    lambda row: row["CD_CAS_LVL_2"] if row["CD_CAS_LVL_1"] == "ACT" else "INAC",
    axis=1
)

print("\nUnique employment values after aggregation:")
print(df_brussels_hc04_1["CD_EMPLOYMENT"].unique())

# Group by sex, employment, and age group, summing population
df_cleaned = df_brussels_hc04_1.groupby(
    ["CD_SEX", "CD_EMPLOYMENT", "CD_AGE_GROUP"], as_index=False
)["MS_POP"].sum()

# Rename age group column to CD_AGE for consistency
df_cleaned.rename(columns={"CD_AGE_GROUP": "CD_AGE"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_HC04_1_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# HC23_2: sex, age (M), education, employment (L), position in family

In [ ]:
# HC23_2
# Filter to get only data for Brussels + group over position in family

# Load HC23_2 data
hc23_2_file = "input_data/TF_CENSUS_2021_HC23_2.xlsx"
df_hc23_2 = pd.read_excel(hc23_2_file)

# Display columns to verify
print("Available columns in HC23_2:")
print(df_hc23_2.columns.tolist())
print(f"\nTotal records: {len(df_hc23_2)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_hc23_2 = df_hc23_2[df_hc23_2["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_hc23_2)}")

# Group by sex, employment, and age group, summing population
df_cleaned = df_brussels_hc23_2.groupby(
    ["CD_SEX", "CD_CAS_LVL_2", "CD_AGE", "CD_EDU"], as_index=False
)["MS_POP"].sum()

# Rename employment column to CD_EMPLOYMENT for consistency
df_cleaned.rename(columns={"CD_CAS_LVL_2": "CD_EMPLOYMENT"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_HC23_2_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# HC05_6: sex, age (M), professional status, work sector (L)

In [ ]:
# HC05_6
# Filter to get only data for Brussels

# Load HC05_6 data
hc05_6_file = "input_data/TF_CENSUS_2021_HC05_6.xlsx"
df_hc05_6 = pd.read_excel(hc05_6_file)

# Display columns to verify
print("Available columns in HC05_6:")
print(df_hc05_6.columns.tolist())
print(f"\nTotal records: {len(df_hc05_6)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_hc05_6 = df_hc05_6[df_hc05_6["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_hc05_6)}")

# Keep only industry, age, sex, professional status fields
# Mapping: CD_IND for industry, CD_AGE for age, CD_SEX for sex, CD_SIE for professional status
df_cleaned = df_brussels_hc05_6[["CD_IND", "CD_AGE", "CD_SEX", "CD_SIE", "MS_POP"]].copy()

# Rename professional status column to CD_PROF_STATUS for consistency
df_cleaned.rename(columns={"CD_SIE": "CD_PROF_STATUS"}, inplace=True)
df_cleaned.rename(columns={"CD_IND": "CD_INDUSTRY"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_HC05_6_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")

# HC05_7: sex, age (L), education, work sector (L)

In [ ]:
# HC05_7
# Filter to get only data for Brussels

# Load HC05_7 data
hc05_7_file = "input_data/TF_CENSUS_2021_HC05_7.xlsx"
df_hc05_7 = pd.read_excel(hc05_7_file)

# Display columns to verify
print("Available columns in HC05_7:")
print(df_hc05_7.columns.tolist())
print(f"\nTotal records: {len(df_hc05_7)}")

# Filter for Brussels province (CD_REFNIS_LVL_1 = 04000)
df_brussels_hc05_7 = df_hc05_7[df_hc05_7["CD_REFNIS_LVL_1"] == 4000].copy()
print(f"Records for Brussels: {len(df_brussels_hc05_7)}")

# Keep only industry, age, sex, professional status fields
# Mapping: CD_IND for industry, CD_AGE for age, CD_SEX for sex, CD_EDU for education
df_cleaned = df_brussels_hc05_7[["CD_IND", "CD_AGE", "CD_SEX", "CD_EDU", "MS_POP"]].copy()

# Rename industry column to CD_INDUSTRY for consistency
df_cleaned.rename(columns={"CD_IND": "CD_INDUSTRY"}, inplace=True)

# Save cleaned data
output_file = "input_data/TF_CENSUS_2021_HC05_7_cleaned.csv"
df_cleaned.to_csv(output_file, index=False)
print(f"\nCleaned data saved to {output_file}")
print(f"Final shape: {df_cleaned.shape}")